# OBS-999 Mismatch between expected and actual physical rotator angle for FBS-driven observations
Author: Bruno Quint

This notebook contains a deeper dive into the analysis associated with the issue described in [OBS-999].  

This ticket addresses a problem with a mismatch between expected and actual physical rotator angles during FBS-driven observations on specific dates.  
The issue began on 2025-05-28 and continued on 2025-05-29, affecting the rotator angles during observations.  
The mismatch disrupted testing for Prompt Processing, as the expected orientation from the nextVisit event was incorrect.  
  
@Lynne Jones suspects the issue may be related to script offsets not being cleared, causing persistent angle discrepancies.  
@Bruno Quint is assigned to investigate the scripts used and address @Te-Wei Tsai's request.  
@Eli Rykoff suggests that pausing and resuming the FBS without clearing the queue might have caused the issue.  
  
Action items include checking the scripts used during the affected period and verifying the commanded rotator positions.  

> So then the question is: can you associate script sal index with a particular visitid (visit id being dayobs + seq_num)? The visit information does not include the script SalIndex (I don't believe -- if you see a link directly from the visit info to script sal index, I would like to know).  However, it does include groupId.
>   
> However, the nextVisit information include both scriptSalIndex and groupId, so using `lsst.sal.ScriptQueue.logevent_nextVisit` you can get both the groupId and the scriptSalIndex in one place and then you can finally link : visitId + groupId -> scriptSalIndex -> observation -> target.

* Can you associate the visitId with the target that resulted in the visit?
* lsst.sal.Scheduler.logevent_target
* lsst.sal.Scheduler.logevent_observation

[OBS-999]: https://rubinobs.atlassian.net/browse/OBS-999

## Initial setup

In [ ]:
day_obs_start = 20250528
day_obs_end = 20250529

# day_obs_start = 20250710
# day_obs_end = 20250712

science_program = "BLOCK-365"

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import re

from astropy.time import Time
from IPython.display import display, HTML

from lsst.summit.utils.efdUtils import (
    makeEfdClient, 
    getEfdData, 
    getDayObsStartTime, 
    getDayObsEndTime
)

# Used to parse ScriptQueue.logevent_nextVisit[rotationSystem]
from lsst.ts.xml.enums import Script as ScriptEnum

# pip install git+https://github.com/lsst-sims/rubin_nights.git
from rubin_nights.connections import get_clients, get_access_token
from rubin_nights.consdb_query import ConsDbTap, ConsDbFastAPI
from rubin_nights import scriptqueue

In [ ]:
# Get the base URL for the queries
api_base = os.getenv("EXTERNAL_INSTANCE_URL", "")

# Get the token for authentication with ConsDB
token = get_access_token()

# Use TAP when creating queries at USDF
consdb_tap = ConsDbTap(api_base=api_base, token=token)

# Use FastAPI when creting queries at the Summit
consdb_fastapi = ConsDbFastAPI(api_base=api_base, auth=('user', token))

# Let's make a uniform interface to simplify the notebook
if "summit" in api_base:
    consdb = consdb_fastapi
elif "usdf" in api_base:
    consdb = consdb_tap
else:
    raise ValueError("Base API does not have Summit nor USDF")

# Get a placeholder for tables
dfs = {}

# Get an efd client
efd_client = makeEfdClient()

# Force pandas to show all the rows
pd.set_option('display.max_rows', None)  

# Colors for the HTMl table
colors = {
    "timestamp": "silver",
    "queue": "royalblue",
    "scripts": "mediumpurple",
}

## Helper Functions

In [ ]:
def camel_to_snake(name):
    s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

## Query visitId

Our ultimate goal is to try to understand where the mismatch rotator angle come from.  
Let's start querying the visits from ConsDB.  
The schema is described at https://sdm-schemas.lsst.io  

The query below will bring down the table with all the visits on a given `dayObs`.

In [ ]:
query = f"select * from cdb_lsstcam.visit1 where science_program = '{science_program}' and day_obs={day_obs_start}"

And the line below performs the query.

In [ ]:
%%time
df_visits = consdb.query(query)

Let's have a peek into one of the entries:

In [ ]:
df_visits.iloc[0]

For this particular case, we need only a subset of the data above:

In [ ]:
selected_cols = ["visit_id", "group_id", "seq_num", "sky_rotation", "obs_start", "obs_end"]
df_visits = df_visits[selected_cols]
df_visits["group_id"] = pd.to_datetime(df_visits["group_id"])
df_visits.sort_values("group_id").head()

In [ ]:
t_start = df_visits.group_id.iloc[0]
t_end = df_visits.group_id.iloc[-1]
print(f"Query other tables from {t_start} to {t_end}")

## Query EFD

Now we have to look at other tables.  
Following Lynne's suggestion, let me dive into a few other dataframes.

In [ ]:
df_target = getEfdData(
    client=efd_client,
    topic="lsst.sal.Scheduler.logevent_target",
    columns="*",
    begin=Time(t_start),
    end=Time(t_end)
)

df_target = df_target.rename(columns={c: camel_to_snake(c) for c in df_target.columns})
selected_cols = ["target_id", "block_id", "sky_angle", "rot_angle", "target_name"]
df_target = df_target[selected_cols].sort_index()
df_target.head()

It is interesting to note that `blockId` seems to be a `salIndex`.  
Is that correct? Is this the `salIndex` of an `add_block` script?
<br>
<br>

In [ ]:
df_observation = getEfdData(
    client=efd_client,
    topic="lsst.sal.Scheduler.logevent_observation",
    columns="*",
    begin=Time(t_start),
    end=Time(t_end)
)

df_observation = df_observation.rename(columns={c: camel_to_snake(c) for c in df_observation.columns})
selected_cols = ["block_id", "rot_sky_pos", "sal_index", "target_id"]
df_observation = df_observation[selected_cols].sort_index()
df_observation.head()

The `targetId` is the closest one to a match.  
`blockId` seems completely inconsistent with the one from `logevent_target`.   
`salIndex` seems to come from the Scheduler sal index.
<br>
<br>

In [ ]:
df_next_visit = getEfdData(
    client=efd_client,
    topic="lsst.sal.ScriptQueue.logevent_nextVisit",
    columns="*",
    begin=Time(t_start),
    end=Time(t_end)
)

# Rename columns to snake_case
df_next_visit = df_next_visit.rename(columns={c: camel_to_snake(c) for c in df_next_visit.columns})
selected_cols = ["camera_angle", "rotation_system", "group_id", "sal_index", "script_sal_index", "survey"]
df_next_visit = df_next_visit[selected_cols].sort_index()

# Ensure group_id is a timestamp
df_next_visit["group_id"] = pd.to_datetime(df_next_visit["group_id"])

# Ensure we have only angles on sky
df_next_visit = df_next_visit[df_next_visit.rotation_system == ScriptEnum.MetadataRotSys.SKY]

# Print out just for curiosity
df_next_visit.head()

The column [rotation_system](https://ts-xml.lsst.io/sal_interfaces/ScriptQueue.html#rotationsystem) tells us in which system the `camera_angle` is. This is mapped in [ts_xml.enums.Script.MetadataRotSky](https://github.com/lsst-ts/ts_xml/blob/797e3d1e07badb5ac7c24e51457c0b3a7fc7cd11/python/lsst/ts/xml/enums/Script.py#L53-L59). In summary, we want all the rows that are `rotation_system == 2`

From consDB, I can get `groupId` which I match with this table too.  
The other two tables do not seem very useful for now.
<br>
<br>

In [ ]:
df_sq_add = getEfdData(
    client=efd_client,
    topic="lsst.sal.ScriptQueue.command_add",
    columns="*",
    begin=Time(t_start),
    end=Time(t_end)
)

df_sq_add = df_sq_add.rename(columns={c: camel_to_snake(c) for c in df_sq_add.columns})
print(df_sq_add.iloc[0].config)

In [ ]:
df_offset = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTPtg.logevent_offsetSummary",
    columns="iaa",
    begin=Time(t_start),
    end=Time(t_end)
)

df_offset = df_offset.rename(columns={c: camel_to_snake(c) for c in df_offset.columns})

In [ ]:
df_sq = getEfdData(
    client=efd_client,
    topic="lsst.sal.ScriptQueue.command_add",
    columns=["path", "config", "locationSalIndex", "block"],
    begin=Time(t_start),
    end=Time(t_end)
)

df_sq = df_sq.rename(columns={c: camel_to_snake(c) for c in df_sq.columns})
df_sq.path.unique()

In [ ]:
def get_rot_sky_from_config(s):

    if "rot_sky" in s.config:
        # Split the configuration per line
        config = s.config.split("\n")
        # Select the line that has the rot_sky parameter
        config = [c for c in config if "rot_sky" in c][0]
        # Get the value
        rot_sky = float(config.split(":")[-1])
        return rot_sky
    
    else: 
        return None


df_sq["rot_sky"] = df_sq.apply(get_rot_sky_from_config, axis=1)

In [ ]:
df_sq.path.unique()

In [ ]:
df_mtptg = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTPtg.currentTargetStatus",
    columns=["timestamp", "parAngle", "demandRot"],
    begin=Time(t_start),
    end=Time(t_end)
)

df_mtptg = df_mtptg.rename(columns={c: camel_to_snake(c) for c in df_mtptg.columns})

## Plot Angles

In [ ]:
def wrap_angle(angle):
    """Convert angle from [0, 360) to [-180, 180)."""
    return (angle + 180) % 360 - 180


title = "Sky Angles per Visit"

fig, ax = plt.subplots(num=title, figsize=[15, 5])

ax.plot(
    df_visits.group_id, 
    wrap_angle(df_visits.sky_rotation), 
    "C0.", 
    label="consdb.visits.sky_rotation",
    markersize=15
)

ax.plot(
    df_offset, 
    "g+", 
    label="MTPtg.logevent_offsetSummary.iaa"
)

ax.plot(
    wrap_angle(df_sq.rot_sky), 
    "yx", 
    label="ScriptQueue.command_add.config"
)

ax.plot(
    df_next_visit.group_id, 
    wrap_angle(df_next_visit.camera_angle), 
    "C3.", 
    label="ScriptQueue.logevent_nextVisit.camera_angle"
)


# ax.plot(
#     df_mtptg.demand_rot,
#     "C4-",
#     label="MTMount.currentTargetStatus.par_angle"
# )


ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
ax.grid(":", alpha=0.2)
ax.legend()
ax.set_xlabel("Group ID [UTC]")
ax.set_ylabel("Angle [deg]")

fig.suptitle(f"{title}\nFrom {day_obs_start} to {day_obs_end}")
fig.autofmt_xdate()  # Inclina datas se precisar

fig.savefig(f"sky_angles_from_{day_obs_start}_to_{day_obs_end}.png")
plt.show()

In [ ]:
print(df_sq[df_sq.path == "maintel/track_target_and_take_image_lsstcam.py"].iloc[0].config)

In [ ]:
df_sq[["path", "block"]]

In [ ]:
title = "

fig, axs = plt.subplots(